In [46]:
from cellxgene_schema.utils import read_h5ad

# Test files for converting from h5ad to cxg
# test.h5ad & test2.h5ad: ok for convert
# pbmc3k.h5ad: not ok for convert # AttributeError: 'dict' object has no attribute 'obs'

# test file
file_path = '/Users/vupha/Documents/Garvan_Data_Science_Platform/single-cell-data-portal/my_data/test.h5ad'
anndata = read_h5ad(file_path, chunk_size=15000)
print(anndata.obs)


                                   Age            ClassAnn  \
CellID                                                       
10X352_1_ABC_1:ACCAACAAGTCCCGACx  10.0  Smooth muscle cell   
10X239_2_AB_1:GACTCAAAGACTCTTGx    8.4  Smooth muscle cell   
10X297_3_AB_1:GACATCAAGCTAGCCCx    7.5  Smooth muscle cell   
10X234_3_AB_1:TTTCCTCTCAAAGACAx    9.0  Smooth muscle cell   
10X234_2_AB_2:GCTGGGTGTGTCCCTTx    9.0  Smooth muscle cell   
...                                ...                 ...   
10X352_3_ABC_1:TTAATCCTCACCGACGx  10.0    ChP Perivascular   
10X239_3_AB_1:GCGAGAAGTGGCCCATx    7.6    ChP Perivascular   
10X297_4_AB_1:GGATGTTTCCAGTGTAx    7.5    ChP Perivascular   
10X352_3_ABC_1:AGCCAGCTCGTCTACCx  10.0    ChP Perivascular   
10X352_3_ABC_1:AGGCATTAGAAGCTCGx  10.0    ChP Perivascular   

                                                    Subclass  \
CellID                                                         
10X352_1_ABC_1:ACCAACAAGTCCCGACx  Smooth muscle cell, mature   
1

In [47]:
# pbmc3k file
file_path = '/Users/vupha/Documents/Garvan_Data_Science_Platform/single-cell-data-portal/my_data/pbmc3k.h5ad'
anndata = read_h5ad(file_path, chunk_size=15000)
print(anndata.obs) # AttributeError: 'dict' object has no attribute 'obs'

AttributeError: 'dict' object has no attribute 'obs'

In [48]:
import h5py # a Python interface to the HDF5 scientific data format

# Compare structure of both files
pbmc3k_path = '/Users/vupha/Documents/Garvan_Data_Science_Platform/single-cell-data-portal/my_data/pbmc3k.h5ad'
test_path = '/Users/vupha/Documents/Garvan_Data_Science_Platform/single-cell-data-portal/my_data/test.h5ad'

print("pbmc3k")
with h5py.File(pbmc3k_path) as f:
    print(f"Top-level keys: {list(f.keys())}")
    for key in f.keys():
        item = f[key]
        print(f"  {key}: {type(item).__name__} - shape: {item.shape if hasattr(item, 'shape') else 'N/A'}")
        if isinstance(item, h5py.Group):
            print(f"    Sub-keys: {list(item.keys())}")

print("\ntest")
with h5py.File(test_path) as f:
    print(f"Top-level keys: {list(f.keys())}")
    for key in f.keys():
        item = f[key]
        print(f"  {key}: {type(item).__name__} - shape: {item.shape if hasattr(item, 'shape') else 'N/A'}")
        if isinstance(item, h5py.Group):
            print(f"    Sub-keys: {list(item.keys())}")

pbmc3k
Top-level keys: ['X', 'obs', 'obsm', 'raw.X', 'raw.var', 'uns', 'var', 'varm']
  X: Dataset - shape: (2638, 1838)
  obs: Dataset - shape: (2638,)
  obsm: Dataset - shape: (2638,)
  raw.X: Group - shape: N/A
    Sub-keys: ['data', 'indices', 'indptr']
  raw.var: Dataset - shape: (13714,)
  uns: Group - shape: N/A
    Sub-keys: ['draw_graph', 'louvain', 'louvain_categories', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups']
  var: Dataset - shape: (1838,)
  varm: Dataset - shape: (1838,)

test
Top-level keys: ['X', 'layers', 'obs', 'obsm', 'obsp', 'uns', 'var', 'varm', 'varp']
  X: Group - shape: N/A
    Sub-keys: ['data', 'indices', 'indptr']
  layers: Group - shape: N/A
    Sub-keys: []
  obs: Group - shape: N/A
    Sub-keys: ['Age', 'CellID', 'CellType', 'ClassAnn', 'KaryotypeFromSouporcell', 'Subclass', 'Superclass', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'cluster_id', 'development_stage', 'development_stage_ontology_term_id', 'd

In [49]:
import scanpy as sc
import h5py
import os
from cellxgene_schema.utils import read_h5ad

# Convert pbmc3k to correct format
pbmc3k_path = '/Users/vupha/Documents/Garvan_Data_Science_Platform/single-cell-data-portal/my_data/pbmc3k.h5ad'
pbmc3k_converted_path = '/Users/vupha/Documents/Garvan_Data_Science_Platform/single-cell-data-portal/my_data/pbmc3k_converted.h5ad'

# Read with scanpy
adata = sc.read_h5ad(pbmc3k_path)

# Write back with scanpy - this will create proper HDF5 Group structure
adata.write_h5ad(pbmc3k_converted_path)

with h5py.File(pbmc3k_converted_path) as f:
    print(f"Top-level keys: {list(f.keys())}")
    for key in f.keys():
        item = f[key]
        print(f"  {key}: {type(item).__name__} - shape: {item.shape if hasattr(item, 'shape') else 'N/A'}")
        if isinstance(item, h5py.Group):
            print(f"    Sub-keys: {list(item.keys())}")


Top-level keys: ['X', 'layers', 'obs', 'obsm', 'obsp', 'raw', 'uns', 'var', 'varm', 'varp']
  X: Dataset - shape: (2638, 1838)
  layers: Group - shape: N/A
    Sub-keys: []
  obs: Group - shape: N/A
    Sub-keys: ['index', 'louvain', 'n_counts', 'n_genes', 'percent_mito']
  obsm: Group - shape: N/A
    Sub-keys: ['X_draw_graph_fr', 'X_pca', 'X_tsne', 'X_umap']
  obsp: Group - shape: N/A
    Sub-keys: ['connectivities', 'distances']
  raw: Group - shape: N/A
    Sub-keys: ['X', 'var', 'varm']
  uns: Group - shape: N/A
    Sub-keys: ['draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups']
  var: Group - shape: N/A
    Sub-keys: ['index', 'n_cells']
  varm: Group - shape: N/A
    Sub-keys: ['PCs']
  varp: Group - shape: N/A
    Sub-keys: []


/opt/homebrew/Caskroom/miniconda/base/envs/cellxgene_annotate/lib/python3.10/site-packages/anndata/compat/__init__.py:371: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(
/opt/homebrew/Caskroom/miniconda/base/envs/cellxgene_annotate/lib/python3.10/site-packages/anndata/compat/__init__.py:371: FutureWarning: Moving element from .uns['neighbors']['connectivities'] to .obsp['connectivities'].

This is where adjacency matrices should go now.
  warn(


In [44]:
# Issues: Missing required metadata fields in pbmc3k.h5ad

# test file
file_path = '/Users/vupha/Documents/Garvan_Data_Science_Platform/single-cell-data-portal/my_data/test.h5ad'
anndata = read_h5ad(file_path, chunk_size=15000)
print(anndata.uns.keys())
print(anndata.uns['schema_version'])
print(anndata.uns['title'])

dict_keys(['batch_condition', 'citation', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'])
7.0.0
Vascular_perivascular


In [45]:
# pbmc3k file
file_path = '/Users/vupha/Documents/Garvan_Data_Science_Platform/single-cell-data-portal/my_data/pbmc3k_converted.h5ad'
anndata = read_h5ad(file_path, chunk_size=15000)
print(anndata.uns.keys()) # KeyError: 'missing Corpora schema field schema_version', 'missing Corpora schema field title'

dict_keys(['draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'])


In [ ]:
import scanpy as sc

# Add required cellxgene schema metadata to pbmc3k_converted
pbmc3k_converted_path = '/Users/vupha/Documents/Garvan_Data_Science_Platform/single-cell-data-portal/my_data/pbmc3k_converted.h5ad'

# Read the converted file
adata = sc.read_h5ad(pbmc3k_converted_path)

# Add required cellxgene schema fields
# https://github.com/chanzuckerberg/single-cell-curation/blob/main/schema/7.0.0/schema.md
adata.uns['schema_version'] = "7.0.0"
adata.uns['title'] = 'PBMC 3K'

# Save the updated file
adata.write_h5ad(pbmc3k_converted_path)

adata_test = read_h5ad(pbmc3k_converted_path, chunk_size=15000)
print(adata_test.uns.keys())
print(adata_test.uns['schema_version'])
print(adata_test.uns['title'])


Current metadata (uns):
dict_keys(['draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups', 'schema_version'])

Adding required cellxgene schema metadata...

✓ Metadata added and saved!

Testing with cellxgene_schema.utils.read_h5ad()...
✓ Successfully read with cellxgene_schema!
  Shape: (2638, 1838)
  Metadata: dict_keys(['draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups', 'schema_version', 'title'])
